# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Veduisback/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I will use Logistic Regression because this lane needs a ranked review queue rather than a complex prediction system. Logistic Regression gives a probability-like score that can rank content items, while its coefficients make the direction of each feature easier to inspect.

The target is `march_had_impressions`, the same future-outcome proxy defined in Week 3. The target uses March information, but the model features use only February information available at the 2026-02-28 decision cutoff. I will treat this as a proxy for a future content outcome, not as a final definition of refresh success.

The five model features are `impressions_30d`, `clicks_30d`, `ctr_30d`, `avg_position_30d`, and `organic_sessions_30d`. The Week-4 baseline will be evaluated on the same content rows and held-out split.

In [4]:
# Section 1: Load decision-time features, future outcome,
# and restrict the modeling population to W04 baseline-eligible content.

import duckdb
import pandas as pd
import numpy as np

from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
print("HF token loaded:", HF_TOKEN is not None)

con = duckdb.connect()

con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")

con.execute("INSTALL azure")
con.execute("LOAD azure")

con.execute(f"""
    CREATE OR REPLACE SECRET hf_secret (
        TYPE HUGGINGFACE,
        TOKEN '{HF_TOKEN}'
    )
""")

DECISION_DATE = "2026-02-28"

FEB_PATH = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-02/*.parquet"
)

MARCH_PATH = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-03/*.parquet"
)

DIM_CONTENT_PATH = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "dim_content.parquet"
)

FEATURE_COLS = [
    "impressions_30d",
    "clicks_30d",
    "ctr_30d",
    "avg_position_30d",
    "organic_sessions_30d",
]

# February decision-time features
feature_sql = f"""
WITH daily AS (
    SELECT
        client_hash_id,
        content_hash_id,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        sessions_organic
    FROM read_parquet('{FEB_PATH}')
    WHERE gsc_data_available IS TRUE
),

performance AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions_30d,
        SUM(gsc_clicks) AS clicks_30d,
        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions)
            ELSE NULL
        END AS ctr_30d,
        AVG(gsc_avg_position) AS avg_position_30d,
        SUM(sessions_organic) AS organic_sessions_30d
    FROM daily
    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT *
FROM performance
"""

features_df = con.execute(feature_sql).df()

# March future-outcome proxy
outcome_sql = f"""
SELECT
    client_hash_id,
    content_hash_id,
    CASE
        WHEN SUM(COALESCE(gsc_impressions, 0)) > 0 THEN 1
        ELSE 0
    END AS march_had_impressions
FROM read_parquet('{MARCH_PATH}')
GROUP BY
    client_hash_id,
    content_hash_id
"""

outcome_df = con.execute(outcome_sql).df()

# W04 eligibility at the decision cutoff.
# This is the exact population used by the W04 baseline.
eligibility_sql = f"""
SELECT
    client_hash_id,
    content_hash_id,
    content_updated_date,
    COALESCE(search_volume, 0) AS search_volume
FROM read_parquet('{DIM_CONTENT_PATH}')
WHERE content_updated_date IS NOT NULL
  AND content_updated_date <= DATE '{DECISION_DATE}'
"""

eligibility_df = con.execute(eligibility_sql).df()

# Combine features + future outcome + W04 eligibility.
model_df = (
    features_df
    .merge(
        outcome_df,
        on=["client_hash_id", "content_hash_id"],
        how="left"
    )
    .merge(
        eligibility_df,
        on=["client_hash_id", "content_hash_id"],
        how="inner"
    )
)

model_df["march_had_impressions"] = (
    model_df["march_had_impressions"]
    .fillna(0)
    .astype(int)
)

# Keep only complete decision-time features.
model_df = model_df.dropna(
    subset=FEATURE_COLS + ["march_had_impressions"]
).copy()

print("W04-eligible model frame shape:", model_df.shape)
print("Unique clients:", model_df["client_hash_id"].nunique())

print("\nTarget distribution:")
display(
    model_df["march_had_impressions"]
    .value_counts()
    .sort_index()
    .rename_axis("march_had_impressions")
    .reset_index(name="n")
)

HF token loaded: True


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

W04-eligible model frame shape: (16309, 10)
Unique clients: 17

Target distribution:


,march_had_impressions,n
0,0,1870
1,1,14439


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I will use a grouped split by `client_hash_id`, with 80% of clients for training and 20% of clients for the held-out test set.

This is more appropriate than a random row split because multiple content items can belong to the same client. Keeping entire clients in one split reduces the chance that the model benefits from seeing other content from the same client during training.

The Week-4 baseline will be evaluated on exactly the same held-out rows. No March outcome information is used to create the split.

In [5]:
from sklearn.model_selection import GroupShuffleSplit

FEATURE_COLS = [
    "impressions_30d",
    "clicks_30d",
    "ctr_30d",
    "avg_position_30d",
    "organic_sessions_30d",
]

TARGET_COL = "march_had_impressions"

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        model_df,
        model_df[TARGET_COL],
        groups=model_df["client_hash_id"]
    )
)

train_df = model_df.iloc[train_idx].copy()
test_df = model_df.iloc[test_idx].copy()

print("Train rows:", len(train_df))
print("Test rows:", len(test_df))
print("Train clients:", train_df["client_hash_id"].nunique())
print("Test clients:", test_df["client_hash_id"].nunique())

client_overlap = set(train_df["client_hash_id"]) & set(test_df["client_hash_id"])
print("Client overlap:", len(client_overlap))

print("\nTrain target rate:")
print(train_df[TARGET_COL].value_counts(normalize=True).sort_index())

print("\nTest target rate:")
print(test_df[TARGET_COL].value_counts(normalize=True).sort_index())

Train rows: 13693
Test rows: 2616
Train clients: 13
Test clients: 4
Client overlap: 0

Train target rate:
march_had_impressions
0    0.122252
1    0.877748
Name: proportion, dtype: float64

Test target rate:
march_had_impressions
0    0.074924
1    0.925076
Name: proportion, dtype: float64


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

I will compare Logistic Regression with the Week-4 `STALE_DEMAND` score on the same held-out content rows.

Because the lane produces a ranked review queue and the positive class is relatively uncommon in the full dataset, I will use Average Precision (AP) as the primary metric. I will also report ROC-AUC as a secondary ranking metric.

The Logistic Regression uses standardized February features and produces a probability for `march_had_impressions = 1`. The Week-4 baseline uses its existing `days_since_update * log(1 + search_volume)` score. Neither method uses March information when producing its ranking.

In [6]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
import pandas as pd
import numpy as np

# ---------------------------------------------------------
# 1. Train Logistic Regression on the training clients
# ---------------------------------------------------------

X_train = train_df[FEATURE_COLS]
y_train = train_df[TARGET_COL]

X_test = test_df[FEATURE_COLS]
y_test = test_df[TARGET_COL]

logreg = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

logreg.fit(X_train, y_train)

model_prob = logreg.predict_proba(X_test)[:, 1]


# ---------------------------------------------------------
# 2. Recreate the exact W04 baseline
# ---------------------------------------------------------

q_baseline = f"""
WITH content AS (
    SELECT
        client_hash_id,
        content_hash_id,
        content_updated_date,
        search_volume,
        DATE_DIFF(
            'day',
            content_updated_date,
            DATE '{DECISION_DATE}'
        ) AS days_since_update
    FROM read_parquet('{DIM_CONTENT_PATH}')
    WHERE content_updated_date IS NOT NULL
      AND content_updated_date <= DATE '{DECISION_DATE}'
),
scored AS (
    SELECT
        client_hash_id,
        content_hash_id,
        content_updated_date,
        days_since_update,
        COALESCE(search_volume, 0) AS search_volume,
        days_since_update
        * LN(1 + COALESCE(search_volume, 0)) AS score
    FROM content
)
SELECT
    client_hash_id,
    content_hash_id,
    score
FROM scored
"""

baseline_scores = con.execute(q_baseline).df()

print("W04 baseline rows:", len(baseline_scores))


# ---------------------------------------------------------
# 3. Join baseline to EXACT held-out model rows
# ---------------------------------------------------------

test_eval = test_df[
    ["client_hash_id", "content_hash_id", TARGET_COL]
].copy()

test_eval["model_score"] = model_prob

test_eval = test_eval.merge(
    baseline_scores,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

print("Held-out model rows:", len(test_df))
print("Rows available for both:", len(test_eval))

assert len(test_eval) == len(test_df), (
    "Baseline/model populations do not match."
)

assert test_eval[
    ["client_hash_id", "content_hash_id"]
].duplicated().sum() == 0


# ---------------------------------------------------------
# 4. Evaluate BOTH methods on the SAME rows
# ---------------------------------------------------------

y_eval = test_eval[TARGET_COL]

baseline_roc_auc = roc_auc_score(
    y_eval,
    test_eval["score"]
)

model_roc_auc = roc_auc_score(
    y_eval,
    test_eval["model_score"]
)

baseline_ap = average_precision_score(
    y_eval,
    test_eval["score"]
)

model_ap = average_precision_score(
    y_eval,
    test_eval["model_score"]
)


comparison = pd.DataFrame({
    "method": [
        "W04 baseline",
        "Logistic Regression"
    ],
    "ROC-AUC": [
        baseline_roc_auc,
        model_roc_auc
    ],
    "Average Precision": [
        baseline_ap,
        model_ap
    ]
})

print("\nModel vs baseline:")
display(comparison.round(4))

print("\nTest positive rate:",
      round(y_eval.mean(), 4))

W04 baseline rows: 135892
Held-out model rows: 2616
Rows available for both: 2616

Model vs baseline:


,method,ROC-AUC,Average Precision
0,W04 baseline,0.4309,0.9109
1,Logistic Regression,0.7179,0.9642



Test positive rate: 0.9251


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

## 4. Errors and interpretation

The target used here is `march_had_impressions`, so model errors should be interpreted as errors on this future outcome proxy rather than as definitive refresh decisions.

I examine two things:

1. Which February features the Logistic Regression model relies on.
2. Where its predictions disagree with the observed March outcome.

Because this is a ranking-oriented task, I also compare the highest-scored pages from the model and the W04 baseline rather than treating a 0.5 classification threshold as the actual editorial decision rule.

In [7]:
# ---------------------------------------------------------
# 1. Inspect Logistic Regression coefficients
# ---------------------------------------------------------

model = logreg.named_steps["model"]

coef_df = pd.DataFrame({
    "feature": FEATURE_COLS,
    "coefficient": model.coef_[0]
})

coef_df["abs_coefficient"] = coef_df["coefficient"].abs()

coef_df = (
    coef_df
    .sort_values("abs_coefficient", ascending=False)
    .reset_index(drop=True)
)

print("Logistic Regression feature coefficients:")
display(coef_df.round(4))


# ---------------------------------------------------------
# 2. Add predictions to held-out evaluation set
# ---------------------------------------------------------

error_df = test_eval.copy()

error_df["predicted_probability"] = error_df["model_score"]

error_df["predicted_class"] = (
    error_df["predicted_probability"] >= 0.5
).astype(int)

error_df["error_type"] = np.select(
    [
        (error_df[TARGET_COL] == 1) &
        (error_df["predicted_class"] == 1),

        (error_df[TARGET_COL] == 0) &
        (error_df["predicted_class"] == 0),

        (error_df[TARGET_COL] == 0) &
        (error_df["predicted_class"] == 1),

        (error_df[TARGET_COL] == 1) &
        (error_df["predicted_class"] == 0),
    ],
    [
        "True Positive",
        "True Negative",
        "False Positive",
        "False Negative",
    ],
    default="Unknown"
)

print("\nPrediction error counts:")
display(
    error_df["error_type"]
    .value_counts()
    .rename_axis("error_type")
    .to_frame("count")
)


# ---------------------------------------------------------
# 3. Compare top-K ranking performance
# ---------------------------------------------------------

def precision_at_k(df, score_col, k):
    ranked = df.sort_values(
        score_col,
        ascending=False
    ).head(k)

    return ranked[TARGET_COL].mean()


k_values = [20, 50, 100]

ranking_rows = []

for k in k_values:
    ranking_rows.append({
        "k": k,
        "W04 baseline precision@k":
            precision_at_k(error_df, "score", k),
        "Logistic Regression precision@k":
            precision_at_k(error_df, "model_score", k),
    })

ranking_comparison = pd.DataFrame(ranking_rows)

print("\nTop-K ranking comparison:")
display(ranking_comparison.round(4))


# ---------------------------------------------------------
# 4. Show examples of model false positives / negatives
# ---------------------------------------------------------

false_positives = (
    error_df[
        error_df["error_type"] == "False Positive"
    ]
    .sort_values("predicted_probability", ascending=False)
)

false_negatives = (
    error_df[
        error_df["error_type"] == "False Negative"
    ]
    .sort_values("predicted_probability", ascending=True)
)

print("\nHighest-confidence false positives:")
display(
    false_positives[
        [
            "client_hash_id",
            "content_hash_id",
            TARGET_COL,
            "predicted_probability",
            "score"
        ]
    ].head(10).round(4)
)

print("\nHighest-confidence false negatives:")
display(
    false_negatives[
        [
            "client_hash_id",
            "content_hash_id",
            TARGET_COL,
            "predicted_probability",
            "score"
        ]
    ].head(10).round(4)
)

Logistic Regression feature coefficients:


,feature,coefficient,abs_coefficient
0,impressions_30d,16.3942,16.3942
1,clicks_30d,3.4579,3.4579
2,organic_sessions_30d,3.0777,3.0777
3,ctr_30d,-0.0352,0.0352
4,avg_position_30d,0.0043,0.0043



Prediction error counts:


,count
error_type,
True Positive,2420
False Positive,196



Top-K ranking comparison:


,k,W04 baseline precision@k,Logistic Regression precision@k
0,20,0.90,0.95
1,50,0.76,0.98
2,100,0.84,0.99



Highest-confidence false positives:


,client_hash_id,content_hash_id,march_had_impressions,predicted_probability,score
439,client_400c21c81c8b46ef,content_204f01b4adb8bd6b,0,1.0000,7.1937
807,client_400c21c81c8b46ef,content_e26125244c6a4d3e,0,0.9985,7.1937
1171,client_400c21c81c8b46ef,content_93e57803443a8422,0,0.9937,7.1937
1183,client_400c21c81c8b46ef,content_c3f15725d95e2fad,0,0.9927,7.1937
1220,client_400c21c81c8b46ef,content_be97056fff5cf5ee,0,0.9910,7.1937
470,client_400c21c81c8b46ef,content_9cee1b432242c949,0,0.9664,7.1937
335,client_400c21c81c8b46ef,content_2e61e64a03f16126,0,0.9490,9.1336
501,client_400c21c81c8b46ef,content_e37c3d3b6e65c628,0,0.9481,7.1937
1985,client_400c21c81c8b46ef,content_a101aef061afd52a,0,0.9403,7.1937
860,client_400c21c81c8b46ef,content_6ccb3ceb692a1559,0,0.9346,7.1937



Highest-confidence false negatives:


,client_hash_id,content_hash_id,march_had_impressions,predicted_probability,score


### Interpretation

The Logistic Regression model is driven primarily by `impressions_30d`, followed by `clicks_30d` and `organic_sessions_30d`. Because the features were standardized before fitting, the coefficient magnitudes are comparable as a rough indication of their contribution to the model's ranking.

The model produced no false negatives or true negatives at a 0.5 classification threshold on this test set. This is largely a consequence of the high positive rate in the held-out population (92.5%), so the binary classification threshold is not the most useful way to evaluate this lane.

The ranking analysis is more aligned with the actual decision. Among the top 20, 50, and 100 pages, Logistic Regression had higher observed positive rates than the W04 baseline on the same held-out rows. In particular, its precision@50 was 0.98 compared with 0.76 for the W04 baseline, while precision@100 was 0.99 compared with 0.84.

The false-positive examples show an important limitation: the model can assign very high probabilities to pages that subsequently have no March impressions. These should not be interpreted as definitively incorrect refresh recommendations because `march_had_impressions` is only a future outcome proxy, not a direct measure of refresh value.

Overall, the model appears useful as a ranking experiment on this held-out client split, but the result should not be treated as proof that Logistic Regression is a superior production refresh strategy. The target proxy and the small number of client groups remain important limitations.

## Self-check

- [x] Compared the model against the W04 baseline.
- [x] Used exactly the same held-out rows for both methods.
- [x] Used a client-grouped split with zero client overlap.
- [x] Kept March information out of the model features.
- [x] Explained why Logistic Regression was selected.
- [x] Reported ROC-AUC and Average Precision.
- [x] Compared ranking performance at top-K review queue sizes.
- [x] Inspected model coefficients and prediction errors.
- [x] Documented the limitation of the `march_had_impressions` target proxy.
- [x] Documented the high positive rate and its effect on threshold-based classification.
- [x] Avoided treating higher model performance as proof of a production-ready refresh strategy.

### Final takeaway

On this held-out client split, Logistic Regression produced stronger ranking metrics than the W04 baseline. The improvement is promising for the modeling experiment, but the target is still a proxy for future impressions rather than a direct measure of refresh value. Further validation with a more decision-aligned target and additional client/time splits would be needed before treating the model as a production ranking system.